# Adult Income

## Câu 4. Tách đặc trưng và nhãn

### Yêu cầu

Tách các thuộc tính đầu vào và thuộc tính quyết định.

### Dữ liệu và phương pháp

Thuộc tính quyết định: income. Sử dụng toàn bộ dữ liệu được cung cấp. Giữ nguyên nguồn train/test sau khi loại dòng trống; nối hai tập để thực hiện quy trình của đề. Tương quan cao được quy ước là |r| ≥ 0,8 giữa các thuộc tính đầu vào; giữ cột đứng trước trong thứ tự gốc.

In [1]:
print("Đặng Ngọc Hiếu - 6451071024")

from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "datasets").is_dir():
    if (ROOT / "6451071024_Lab2" / "datasets").is_dir():
        ROOT = ROOT / "6451071024_Lab2"
        break
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Không tìm thấy thư mục datasets của LAB2.")
    ROOT = ROOT.parent
pd.set_option("display.max_columns", None)


train_raw = pd.read_csv(ROOT / "datasets/adult/adult.data.csv", skipinitialspace=True)
test_raw = pd.read_csv(ROOT / "datasets/adult/adult.test.csv", skipinitialspace=True)
if "income" not in train_raw or "income" not in test_raw:
    raise ValueError("Hai tệp Adult phải có cột income.")
if "fnlwgt" not in train_raw or "fnlwgt" in test_raw:
    raise ValueError("Kiểm tra hai phiên bản Adult: fnlwgt chỉ có trong tập huấn luyện.")


Đặng Ngọc Hiếu - 6451071024


In [2]:
print("Đặng Ngọc Hiếu - 6451071024")

# Kiểu string tường minh giúp nhận diện thiếu mà không dựa vào downcasting.
def clean_adult(frame):
    frame = frame.copy()
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        values = frame[column].astype("string").str.strip()
        frame[column] = values.mask(values.isin(["?", ""]))
    return frame.dropna()

train = clean_adult(train_raw).drop(columns="fnlwgt")
test = clean_adult(test_raw)
if list(train.columns) != list(test.columns):
    raise ValueError("Tên và thứ tự cột Adult chưa khớp sau khi bỏ fnlwgt.")
n_train = len(train)
data = pd.concat([train, test], ignore_index=True)
TARGET = "income"


Đặng Ngọc Hiếu - 6451071024


In [3]:
print("Đặng Ngọc Hiếu - 6451071024")

# Tái hiện bước chọn thuộc tính của câu 3, không vẽ lại heatmap.
corr = data.drop(columns=TARGET).select_dtypes(include="number").corr(method="pearson")
CORRELATION_THRESHOLD = 0.8
# |r| >= 0,8 là quy ước đã nêu trong phương pháp, không phải ngưỡng đề ấn định.
drop_columns = []
for j, column in enumerate(corr.columns):
    for previous in corr.columns[:j]:
        value = corr.loc[previous, column]
        if previous not in drop_columns and pd.notna(value) and abs(value) >= CORRELATION_THRESHOLD:
            drop_columns.append(column)
            break
data = data.drop(columns=drop_columns)


Đặng Ngọc Hiếu - 6451071024


In [4]:
print("Đặng Ngọc Hiếu - 6451071024")

features = data.drop(columns=TARGET)
labels = data[TARGET].copy()
if labels.isna().any() or features.empty:
    raise ValueError("Dữ liệu cần có đặc trưng và nhãn không trống.")
print("Thuộc tính quyết định:", TARGET)
print("Kích thước features / labels:", features.shape, labels.shape)
display(features.head(), labels.head().to_frame())


Đặng Ngọc Hiếu - 6451071024
Thuộc tính quyết định: income
Kích thước features / labels: (45222, 13) (45222,)


,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba


,income
0,<=50K
1,<=50K
2,<=50K
3,<=50K
4,<=50K
